# 🤖 GPT Smart Assistant with Code Execution


**قابلیت‌ها:**
- تشخیص اینکه سوال نیاز به محاسبات ریاضی داره یا نه
- تولید کد پایتون و اجرای خودکار اون
- retry خودکار در صورت خطا (تا ۳ بار)
- تولید پاسخ نهایی انسانی و طبیعی

## ۱. نصب و import کتابخانه‌ها

In [1]:
# !pip install openai dotenv
from dotenv import load_dotenv
import io
import sys
import os
from openai import OpenAI

## ۲. تنظیم API Key

In [2]:
load_dotenv()
client = OpenAI()


MODEL = "gpt-4o-mini"

## ۳. توابع اصلی

In [12]:
def check_if_needs_math(prompt: str) -> bool:
    """Ask GPT whether this question requires mathematical calculation or not."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": "فقط با YES یا NO جواب بده. هیچ توضیح اضافه‌ای نده."
            },
            {
                "role": "user",
                "content": f"آیا سوال زیر نیاز به محاسبات ریاضی یا اجرای کد پایتون دارد؟\n\nسوال: {prompt}"
            }
        ],
        max_tokens=10,
        temperature=0
    )
    answer = response.choices[0].message.content.strip().upper()
    return "YES" in answer


def execute_code_safely(code: str):
    """Safely execute Python code and return the output."""
    old_stdout = sys.stdout
    sys.stdout = buffer = io.StringIO()

    # Remove Markdown code fences from the code.
    clean_code = code.strip()
    for marker in ["```python", "```Python", "```", "python"]:
        clean_code = clean_code.replace(marker, "")
    clean_code = clean_code.strip()

    error = None
    try:
        exec(clean_code, {})
    except Exception as e:
        error = str(e)
    finally:
        sys.stdout = old_stdout

    output = buffer.getvalue()
    return output, error, clean_code


def generate_code(prompt: str, error_context: str = None, broken_code: str = None) -> str:
    """Ask GPT to generate Python code (or fix the error)."""
    if error_context and broken_code:
        user_message = f"""The following code has this error: {error_context}

Code:
{broken_code}

Main question: {prompt}

Fix the code."""
    else:
        user_message = prompt + "Write only Python code. Do not provide any explanation. Use print() to display the result."

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": "You are a Python programmer. Write only pure Python code, without any explanation or markdown. Always use print() to display the result."
            },
            {"role": "user", "content": user_message}
        ],
        temperature=0.2
    )
    return response.choices[0].message.content


def generate_human_response(original_question: str, code: str, output: str) -> str:
    """Generate a natural, human-like response based on the result of the code execution"""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": "You are an intelligent assistant who must respond naturally and humanly, without showing code or technical details."
            },
            {
                "role": "user",
                "content": f"""User question: {original_question}

Calculation result: {output}

Based on the result, provide a complete, clear, and human-like response. Only give the final answer."""
            }
        ],
        temperature=0.7
    )
    return response.choices[0].message.content


def generate_text_response(prompt: str) -> str:
    """پاسخ معمولی بدون نیاز به کد"""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are an intelligent and helpful assistant."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.7
    )
    return response.choices[0].message.content


print("✅ تمام توابع آماده‌ست!")

✅ تمام توابع آماده‌ست!


## ۴. تابع اصلی Smart Assistant

In [5]:
def smart_assistant(prompt: str, max_attempts: int = 3, verbose: bool = True):
    """
    دستیار هوشمند که:
    - تشخیص میده سوال ریاضیه یا نه
    - در صورت نیاز کد تولید و اجرا میکنه
    - پاسخ انسانی تولید میکنه

    Args:
        prompt: سوال کاربر
        max_attempts: حداکثر تلاش برای fix کردن کد
        verbose: نمایش جزئیات اجرا

    Returns:
        dict: شامل پاسخ، کد (اگه بود)، و خروجی کد
    """
    print(f"\n{'='*60}")
    print(f"❓ سوال: {prompt}")
    print('='*60)

    # مرحله ۱: تشخیص نوع سوال
    print("🔍 در حال بررسی نوع سوال...")
    needs_math = check_if_needs_math(prompt)

    if needs_math:
        print("🔢 این سوال نیاز به محاسبات ریاضی دارد.")

        attempt = 0
        error_context = None
        broken_code = None

        while attempt < max_attempts:
            attempt += 1
            print(f"\n⚙️  تلاش {attempt} از {max_attempts}...")

            # تولید کد
            raw_code = generate_code(prompt, error_context, broken_code)

            # اجرای کد
            output, error, clean_code = execute_code_safely(raw_code)

            if verbose:
                print(f"📝 کد تولید شده:\n{'-'*40}\n{clean_code}\n{'-'*40}")

            if error:
                print(f"⚠️  خطا: {error}")
                error_context = error
                broken_code = clean_code
            else:
                print(f"✅ محاسبات با موفقیت انجام شد!")
                print(f"📊 خروجی کد: {output.strip()}")

                # تولید پاسخ انسانی
                print("\n💬 در حال تولید پاسخ نهایی...")
                human_answer = generate_human_response(prompt, clean_code, output)

                print(f"\n{'='*60}")
                print("💬 پاسخ نهایی:")
                print(human_answer)
                print('='*60)

                return {
                    "answer": human_answer,
                    "code": clean_code,
                    "output": output,
                    "attempts": attempt,
                    "type": "math"
                }

        print("❌ متاسفانه پس از چند تلاش نتوانستیم مسئله را حل کنیم.")
        return {"answer": None, "error": "Max attempts reached", "type": "math"}

    else:
        # سوال معمولی
        print("💬 در حال پاسخ دادن...")
        answer = generate_text_response(prompt)

        print(f"\n{'='*60}")
        print("💬 پاسخ:")
        print(answer)
        print('='*60)

        return {
            "answer": answer,
            "code": None,
            "output": None,
            "type": "text"
        }

print("✅ تابع smart_assistant آماده‌ست!")

✅ تابع smart_assistant آماده‌ست!


## ۵. تست با مثال‌های مختلف

In [6]:
# تست ۱: سوال ریاضی ساده
result = smart_assistant("ریشه دوم ۱۴۴ چقدر است؟")


❓ سوال: ریشه دوم ۱۴۴ چقدر است؟
🔍 در حال بررسی نوع سوال...
🔢 این سوال نیاز به محاسبات ریاضی دارد.

⚙️  تلاش 1 از 3...
📝 کد تولید شده:
----------------------------------------
print(144**0.5)
----------------------------------------
✅ محاسبات با موفقیت انجام شد!
📊 خروجی کد: 12.0

💬 در حال تولید پاسخ نهایی...

💬 پاسخ نهایی:
ریشه دوم ۱۴۴ برابر با ۱۲ است.


In [7]:
# تست ۲: سوال ریاضی پیچیده‌تر
result = smart_assistant("اگه ۱۵ درصد تخفیف روی قیمت ۲۵۰,۰۰۰ تومان بخوریم، قیمت نهایی چقدر میشه؟")


❓ سوال: اگه ۱۵ درصد تخفیف روی قیمت ۲۵۰,۰۰۰ تومان بخوریم، قیمت نهایی چقدر میشه؟
🔍 در حال بررسی نوع سوال...
🔢 این سوال نیاز به محاسبات ریاضی دارد.

⚙️  تلاش 1 از 3...
📝 کد تولید شده:
----------------------------------------
price = 250000
discount = 0.15
final_price = price * (1 - discount)
print(final_price)
----------------------------------------
✅ محاسبات با موفقیت انجام شد!
📊 خروجی کد: 212500.0

💬 در حال تولید پاسخ نهایی...

💬 پاسخ نهایی:
با اعمال ۱۵ درصد تخفیف روی قیمت ۲۵۰,۰۰۰ تومان، قیمت نهایی ۲۱۲,۵۰۰ تومان خواهد بود.


In [8]:
# تست ۳: سوال آماری
result = smart_assistant("میانگین اعداد [23, 45, 12, 67, 89, 34] چقدر است؟")


❓ سوال: میانگین اعداد [23, 45, 12, 67, 89, 34] چقدر است؟
🔍 در حال بررسی نوع سوال...
🔢 این سوال نیاز به محاسبات ریاضی دارد.

⚙️  تلاش 1 از 3...
📝 کد تولید شده:
----------------------------------------
numbers = [23, 45, 12, 67, 89, 34]
mean = sum(numbers) / len(numbers)
print(mean)
----------------------------------------
✅ محاسبات با موفقیت انجام شد!
📊 خروجی کد: 45.0

💬 در حال تولید پاسخ نهایی...

💬 پاسخ نهایی:
میانگین اعداد [23, 45, 12, 67, 89, 34] برابر با 45.0 است.


In [9]:
# تست ۴: سوال غیر ریاضی
result = smart_assistant("پایتخت فرانسه کجاست؟")


❓ سوال: پایتخت فرانسه کجاست؟
🔍 در حال بررسی نوع سوال...
💬 در حال پاسخ دادن...

💬 پاسخ:
پایتخت فرانسه، پاریس است.


In [10]:
# تست ۵: سوال برنامه‌نویسی با محاسبه
result = smart_assistant("فاکتوریل عدد ۱۰ را حساب کن")


❓ سوال: فاکتوریل عدد ۱۰ را حساب کن
🔍 در حال بررسی نوع سوال...
🔢 این سوال نیاز به محاسبات ریاضی دارد.

⚙️  تلاش 1 از 3...
📝 کد تولید شده:
----------------------------------------
result = 1
for i in range(1, 11):
    result *= i
print(result)
----------------------------------------
✅ محاسبات با موفقیت انجام شد!
📊 خروجی کد: 3628800

💬 در حال تولید پاسخ نهایی...

💬 پاسخ نهایی:
فاکتوریل عدد ۱۰ برابر با ۳۶۲۸۸۰۰ است.
